In [1]:
import pyomo.environ as pyo
import pandas as pd
import math
import numpy as np
from collections import defaultdict
from datetime import timedelta
from pyomo.util.infeasible import log_infeasible_constraints
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
class charging_point():
    def __init__(self, name, ev_capacity, ev_max_power, ev_arrival_soc, ev_arrival, ev_departure, ev_desired_soc):
        self.efficiency = 0.93
        # Input validation using assertions
        assert isinstance(ev_capacity, list), "ev_capacity must be a list"
        assert isinstance(ev_max_power, list), "ev_max_power must be a list"
        assert isinstance(ev_arrival_soc, list), "ev_arrival_soc must be a list"
        assert isinstance(ev_arrival, list), "ev_arrival must be a list"
        assert isinstance(ev_departure, list), "ev_departure must be a list"
        assert isinstance(ev_desired_soc, list), "ev_desired_soc must be a list"

        assert len(ev_arrival) == len(ev_departure), "ev_arrival and ev_departure must have the same length"
        assert len(ev_arrival) == len(ev_desired_soc), "ev_arrival and ev_desired_soc must have the same length"
        assert len(ev_arrival_soc) == len(ev_desired_soc), "ev_arrival_soc and ev_desired_soc must have the same length"
        assert len(ev_capacity) == len(ev_desired_soc), "ev_capacity and ev_desired_soc must have the same length"
        assert len(ev_max_power) == len(ev_desired_soc), "ev_max_power and ev_desired_soc must have the same length"

        for i in range(len(ev_arrival)):
            assert ev_arrival[i] < ev_departure[i], f"ev_arrival[{i}] must be less than ev_departure[{i}]"
            assert 0.2 <= ev_arrival_soc[i] <= 1, f"ev_arrival_soc[{i}] must be between 0.2 and 1"
            assert 0 <= ev_desired_soc[i] <= 1, f"ev_desired_soc[{i}] must be between 0 and 1"
            min_time_to_charge = ev_departure[i] - ev_arrival[i]
            min_req_charge = (ev_desired_soc[i] - ev_arrival_soc[i]) * ev_capacity[i] / self.efficiency
            min_req_charge_per_time = min_req_charge / min_time_to_charge
            assert min_req_charge_per_time <= ev_max_power[i], f"min_req_charge_per_time ({min_req_charge_per_time:.2f}) must be less than or equal to ev_max_power[{i}] ({ev_max_power[i]}) for {name}"

        self.name = name
        self.ev_capacity = ev_capacity
        self.ev_max_power = ev_max_power
        self.ev_arrival_soc = ev_arrival_soc
        self.ev_desired_soc = ev_desired_soc
        self.ev_arrival = ev_arrival
        self.ev_departure = ev_departure
        self.num_evs = len(ev_capacity) # Store the number of EVs

        #EV aging parameters:
        self.BatVol = 400 #Battery voltage in V
        self.eoi = 0.2 # End-of-life of battery in percentage (20%)
        self.SalRep = 0.5 # Salvation value over replacement value
        self.Rep = [11.1*137*i for i in self.ev_capacity] # replacement value of battery (SEK/kWh)
        self.discount = 0.05 # discount rate (5%)
        self.life = 10 # lifetime of battery (10 years)
        self.OM = [0.02 * i for i in self.Rep] # operation and maintenance cost of battery (2% of replacement cost)

class building():
    def __init__(self, name, load, pv_production, bess_capacity, bess_max_power, bess_initial_soc):
        self.efficiency = 0.93
        self.name = name
        self.load = load
        self.pv_production = pv_production
        self.bess_capacity = bess_capacity
        self.bess_max_power = bess_max_power
        self.bess_initial_soc = bess_initial_soc
        
        #BESS aging parameters:
        self.bess_BatVol = 400 #Battery voltage in V
        self.bess_eoi = 0.2 # End-of-life of battery in percentage (20%)
        self.bess_SalRep = 0.5 # Salvation value over replacement value
        self.bess_Rep = 11.1*137*self.bess_capacity # replacement value of battery (SEK/kWh)
        self.bess_discount = 0.05 # discount rate (5%)
        self.bess_life = 10 # lifetime of battery (10 years)
        self.bess_OM = 0.02 *self.bess_Rep # operation and maintenance cost of battery (2% of replacement cost)

class LEC_Opt_spot():
    def __init__(self, charging_points, buildings, spot_prices, temperature, previous_monthly_peak=0, v2g_on=1, incentive_per_kwh=0.1):
        self.M = 10000
        self.charging_points = charging_points
        self.buildings = buildings
        self.spot_prices = spot_prices
        self.temperature = temperature
        self.incentive_per_kwh = incentive_per_kwh
        self.v2g_on = v2g_on
        self.previous_monthly_peak = previous_monthly_peak
        self.model = pyo.ConcreteModel()
        self.build_model()

    def build_model(self):
        self.model.T = pyo.Set(initialize=range(len(self.spot_prices)))
        self.model.spot_prices = self.spot_prices
        self.model.temperature = self.temperature
        self.model.previous_monthly_peak = self.previous_monthly_peak
        self.model.monthly_peak = pyo.Var(initialize = 0)
        self.model.P_im_grid = pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 100000))
        self.model.P_ex_grid = pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 100000))
        self.model.B_im_grid = pyo.Var(self.model.T, within=pyo.Binary)
        self.model.Peakload = pyo.Var(within=pyo.NonNegativeReals)
        self.model.transmission_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.supplier_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.overall_dso_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.tax_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.peak_cost = pyo.Var(self.model.T, within=pyo.Reals)
        self.model.Subscription_fee = 605/30  # Subscription fee SEK/14 days
        self.model.Transmission_fee = 0.113   # Electricity transmission fee SEK/kWh
        self.model.Transmission_health_incentive = 0.04 #Transmission health incentive SEK/kWh
        self.model.Effect_fee = 61.55/30       # Effect fee SEK/kW/14 days
        self.model.Energy_tax = 0.439           # Tax fee SEK/kWh
        self.model.Energy_certificate = 0.005   #Energy certificate SEK/kWh
        self.model.compensation_fee = 0.02    # Transfer compensation fee SEK/kWh

        for charge_point in self.charging_points:
            setattr(self.model, f'{charge_point.name}_P', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{charge_point.name}_Cyclic_cost', pyo.Var(self.model.T, within = pyo.Reals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{charge_point.name}_Calendar_cost', pyo.Var(self.model.T, within = pyo.Reals, bounds=(-1000000, 1000000), initialize = 0))

            # Iterate through each EV at the charging point
            for ev_index in range(charge_point.num_evs):
                ev_name = f'{charge_point.name}_ev{ev_index}'  # Unique name for each EV
                setattr(self.model, f'{ev_name}_ch', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, charge_point.ev_max_power[ev_index])))
                setattr(self.model, f'{ev_name}_ds', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, self.v2g_on * charge_point.ev_max_power[ev_index])))
                setattr(self.model, f'{ev_name}_soc', pyo.Var(self.model.T, within=pyo.NonNegativeReals, bounds=(0, 1)))
                setattr(self.model, f'{ev_name}_Bch', pyo.Var(self.model.T, within=pyo.Binary))

                #Aging decision variables:
                setattr(self.model, f'{ev_name}_Ba', pyo.Var(self.model.T, within=pyo.Binary))
                setattr(self.model, f'{ev_name}_Bb', pyo.Var(self.model.T, within=pyo.Binary))
                setattr(self.model, f'{ev_name}_Bc', pyo.Var(self.model.T, within=pyo.Binary))
                setattr(self.model, f'{ev_name}_Ka', pyo.Var(self.model.T, within=pyo.NonNegativeReals))
                setattr(self.model, f'{ev_name}_Kb', pyo.Var(self.model.T, within=pyo.NonNegativeReals))
                setattr(self.model, f'{ev_name}_Kc', pyo.Var(self.model.T, within=pyo.NonNegativeReals))
                setattr(self.model, f'{ev_name}_CalAg', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0))
                setattr(self.model, f'{ev_name}_CycAg', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0))
                setattr(self.model, f'{ev_name}_CalCost', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0))
                setattr(self.model, f'{ev_name}_CycCost', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0))

                ev_capacity = charge_point.ev_capacity[ev_index]
                ev_arrival_soc = charge_point.ev_arrival_soc[ev_index]
                ev_arrival = charge_point.ev_arrival[ev_index]
                ev_departure = charge_point.ev_departure[ev_index]
                ev_desired_soc = charge_point.ev_desired_soc[ev_index]
                ev_voltage = charge_point.BatVol
                ev_SalRep = charge_point.SalRep
                ev_Rep = charge_point.Rep[ev_index]
                ev_discount = charge_point.discount
                ev_life = charge_point.life
                ev_OM = charge_point.OM[ev_index]
                ev_eoi = charge_point.eoi

                def ev_soc_rule(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    if t == ev_arrival:
                        return ev_soc == ev_arrival_soc + (ev_ch * charge_point.efficiency - ev_ds / charge_point.efficiency) / ev_capacity
                    elif ev_arrival < t <= ev_departure:
                        ev_previous_soc = getattr(model, f'{ev_name}_soc')[t - 1]
                        return ev_soc == ev_previous_soc + (ev_ch * charge_point.efficiency - ev_ds / charge_point.efficiency) / ev_capacity
                    else:
                        return pyo.Constraint.Skip
                setattr(self.model, f'{ev_name}_soc_constraint', pyo.Constraint(self.model.T, rule=ev_soc_rule))

                def ev_soc_min_rule(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    ev_arrival = charge_point.ev_arrival[ev_index]
                    ev_departure = charge_point.ev_departure[ev_index]
                    if ev_arrival <= t <= ev_departure:
                        return ev_soc >= 0.2
                    return ev_soc == 0
                setattr(self.model, f'{ev_name}_soc_min_constraint', pyo.Constraint(self.model.T, rule=ev_soc_min_rule))

                def ev_max_ch(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_Bch = getattr(model, f'{ev_name}_Bch')[t]
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    return ev_ch <= ev_Bch * charge_point.ev_max_power[ev_index]
                setattr(self.model, f'{ev_name}_max_ch_constraint', pyo.Constraint(self.model.T, rule=ev_max_ch))

                def ev_max_ds(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_Bch = getattr(model, f'{ev_name}_Bch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    return ev_ds <= (1 - ev_Bch) * charge_point.ev_max_power[ev_index]
                setattr(self.model, f'{ev_name}_max_ds_constraint', pyo.Constraint(self.model.T, rule=ev_max_ds))

                def ev_avail_ch(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    if charge_point.ev_arrival[ev_index] < t < charge_point.ev_departure[ev_index]:
                        return ev_ch >= 0
                    return ev_ch == 0
                setattr(self.model, f'{ev_name}_avail_ch_constraint', pyo.Constraint(self.model.T, rule=ev_avail_ch))

                def ev_avail_ds(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    if charge_point.ev_arrival[ev_index] < t < charge_point.ev_departure[ev_index]:
                        return ev_ds >= 0
                    return ev_ds == 0
                setattr(self.model, f'{ev_name}_avail_ds_constraint', pyo.Constraint(self.model.T, rule=ev_avail_ds))

                def ev_desired_soc(model, t, charge_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    if t == charge_point.ev_departure[ev_index]:
                        return ev_soc >= charge_point.ev_desired_soc[ev_index]
                    return pyo.Constraint.Skip
                setattr(self.model, f'{ev_name}_desired_soc_constraint', pyo.Constraint(self.model.T, rule=ev_desired_soc))

                #Aging constraints:
                def ev_calendar_aging(model, t, charging_point=charge_point, ev_index=ev_index):
                    calendar_aging = getattr(model, f'{ev_name}_CalAg')[t]
                    Ka = getattr(model, f'{ev_name}_Ka')[t]
                    Ba = getattr(model, f'{ev_name}_Ba')[t]
                    Kb = getattr(model, f'{ev_name}_Kb')[t]
                    Bb = getattr(model, f'{ev_name}_Bb')[t]
                    Kc = getattr(model, f'{ev_name}_Kc')[t]
                    Bc = getattr(model, f'{ev_name}_Bc')[t]
                    day = 1 #dummy day
                    return calendar_aging == 0.01*((36.7*Ka + 1224.6*Ba) + (168.7*Kb + 3103.7*Bb) + (41.9*Kc + 6265.2*Bc))*(math.exp(-24500/(model.temperature[t]*8.314))*0.5*0.042)/((day+90)**0.5)
                setattr(self.model, f'{ev_name}_calendar_aging_constraint', pyo.Constraint(self.model.T, rule=ev_calendar_aging))

                def cal_aging_2(model, t, charging_point=charge_point, ev_index=ev_index):
                    Ba = getattr(model, f'{ev_name}_Ba')[t]
                    Bb = getattr(model, f'{ev_name}_Bb')[t]
                    Bc = getattr(model, f'{ev_name}_Bc')[t]
                    return Ba + Bb + Bc == 1
                setattr(self.model, f'{ev_name}_cal_aging_2_contraint', pyo.Constraint(self.model.T, rule=cal_aging_2))

                def cal_aging_3(model, t, charging_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    Ba = getattr(model, f'{ev_name}_Ba')[t]
                    Bb = getattr(model, f'{ev_name}_Bb')[t]
                    Bc = getattr(model, f'{ev_name}_Bc')[t]
                    return ev_soc >= ((0.5*Bb) - (self.M*Ba) + (0.7*Bc))
                setattr(self.model, f'{ev_name}_cal_aging_3_constraint', pyo.Constraint(self.model.T, rule=cal_aging_3))

                def cal_aging_4(model, t, charging_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    Ba = getattr(model, f'{ev_name}_Ba')[t]
                    Bb = getattr(model, f'{ev_name}_Bb')[t]
                    Bc = getattr(model, f'{ev_name}_Bc')[t]
                    return ev_soc <= ((0.5*Ba) + (self.M*Bc) + (0.7*Bb))
                setattr(self.model, f'{ev_name}_cal_aging_4_constraint', pyo.Constraint(self.model.T, rule=cal_aging_4))

                def cal_aging_5(model, t, charging_point=charge_point, ev_index=ev_index):
                    Ba = getattr(model, f'{ev_name}_Ba')[t]
                    Ka = getattr(model, f'{ev_name}_Ka')[t]
                    return Ka <= self.M*Ba
                setattr(self.model, f'{ev_name}_cal_aging_5_constraint', pyo.Constraint(self.model.T, rule=cal_aging_5))

                def cal_aging_6(model, t, charging_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    Ka = getattr(model, f'{ev_name}_Ka')[t]
                    return Ka <= ev_soc
                setattr(self.model, f'{ev_name}_cal_aging_6_constraint', pyo.Constraint(self.model.T, rule=cal_aging_6))

                def cal_aging_7(model, t, charging_point=charge_point, ev_index=ev_index):
                    Ba = getattr(model, f'{ev_name}_Ba')[t]
                    Ka = getattr(model, f'{ev_name}_Ka')[t]
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    return (ev_soc - (1-Ba) * self.M) <= Ka
                setattr(self.model, f'{ev_name}_cal_aging_7_constraint', pyo.Constraint(self.model.T, rule=cal_aging_7))

                def cal_aging_8(model, t, charging_point=charge_point, ev_index=ev_index):
                    Bb = getattr(model, f'{ev_name}_Bb')[t]
                    Kb = getattr(model, f'{ev_name}_Kb')[t]
                    return Kb <= self.M*Bb
                setattr(self.model, f'{ev_name}_cal_aging_8_constraint', pyo.Constraint(self.model.T, rule=cal_aging_8))

                def cal_aging_9(model, t, charging_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    Kb = getattr(model, f'{ev_name}_Kb')[t]
                    return Kb <= ev_soc
                setattr(self.model, f'{ev_name}_cal_aging_9_constraint', pyo.Constraint(self.model.T, rule=cal_aging_9))

                def cal_aging_10(model, t, charging_point=charge_point, ev_index=ev_index):
                    Bb = getattr(model, f'{ev_name}_Bb')[t]
                    Kb = getattr(model, f'{ev_name}_Kb')[t]
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    return (ev_soc - (1-Bb) * self.M) <= Kb
                setattr(self.model, f'{ev_name}_cal_aging_10_constraint', pyo.Constraint(self.model.T, rule=cal_aging_10))

                def cal_aging_11(model, t, charging_point=charge_point, ev_index=ev_index):
                    Bc = getattr(model, f'{ev_name}_Bc')[t]
                    Kc = getattr(model, f'{ev_name}_Kc')[t]
                    return Kc <= self.M*Bc
                setattr(self.model, f'{ev_name}_cal_aging_11_constraint', pyo.Constraint(self.model.T, rule=cal_aging_11))

                def cal_aging_12(model, t, charging_point=charge_point, ev_index=ev_index):
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    Kc = getattr(model, f'{ev_name}_Kc')[t]
                    return Kc <= ev_soc
                setattr(self.model, f'{ev_name}_cal_aging_12_constraint', pyo.Constraint(self.model.T, rule=cal_aging_12))

                def cal_aging_13(model, t, charging_point=charge_point, ev_index=ev_index):
                    Bc = getattr(model, f'{ev_name}_Bc')[t]
                    Kc = getattr(model, f'{ev_name}_Kc')[t]
                    ev_soc = getattr(model, f'{ev_name}_soc')[t]
                    return (ev_soc - (1-Bc) * self.M) <= Kc
                setattr(self.model, f'{ev_name}_cal_aging_13_constraint', pyo.Constraint(self.model.T, rule=cal_aging_13))

                def ev_calendar_aging_cost(model, t, charging_point=charge_point, ev_index=ev_index):
                    calendar_aging = getattr(model, f'{ev_name}_CalAg')[t]
                    calendar_aging_cost = getattr(model, f'{ev_name}_CalCost')[t]
                    return calendar_aging_cost == (((1-ev_SalRep)*ev_Rep/((1+ev_discount)**ev_life)+ev_OM\
                                                *(((1+ev_discount)**ev_life)-1)/(ev_discount*(1+ev_discount)**ev_life))\
                                                /(ev_eoi))*(calendar_aging)
                setattr(self.model, f'{ev_name}_calendar_aging_cost_constraint', pyo.Constraint(self.model.T, rule=ev_calendar_aging_cost))

                def ev_cyclic_aging(model, t, charging_point=charge_point, ev_index=ev_index):
                    cyclic_aging = getattr(model, f'{ev_name}_CycAg')[t]
                    ev_ch = getattr(model, f'{ev_name}_ch')[t]
                    ev_ds = getattr(model, f'{ev_name}_ds')[t]
                    return cyclic_aging == (0.01*(((0.0000086*(self.temperature[t]**2)-0.0051*self.temperature[t]+0.763)*\
                                                      (67.15*ev_ch+67.15*ev_ds-2.94))/(ev_voltage*ev_capacity)))
                setattr(self.model, f'{ev_name}_cyclic_aging_constraint', pyo.Constraint(self.model.T, rule=ev_cyclic_aging))

                def ev_cyclic_aging_cost(model, t, charging_point=charge_point, ev_index=ev_index):
                    cyclic_aging_cost = getattr(model, f'{ev_name}_CycCost')[t]
                    cyclic_aging = getattr(model, f'{ev_name}_CycAg')[t]
                    return cyclic_aging_cost == (((1-ev_SalRep)*ev_Rep/((1+ev_discount)**ev_life)+ev_OM\
                                                *(((1+ev_discount)**ev_life)-1)/(ev_discount*(1+ev_discount)**ev_life))\
                                                /(ev_eoi))*(cyclic_aging)
                setattr(self.model, f'{ev_name}_cyclic_aging_cost_constraint', pyo.Constraint(self.model.T, rule=ev_cyclic_aging_cost))

            def consumption(model, t, charge_point=charge_point):
                P = getattr(model, f'{charge_point.name}_P')[t]
                ev_power = sum(getattr(model, f'{charge_point.name}_ev{ev_index}_ch')[t] - getattr(model, f'{charge_point.name}_ev{ev_index}_ds')[t] for ev_index in range(charge_point.num_evs))
                return  ev_power == P
            setattr(self.model, f'{charge_point.name}_consumption_constraint', pyo.Constraint(self.model.T, rule=consumption))

            def cyclic_cost(model, t, charge_point = charge_point):
                cyclic_cost = getattr(model, f'{charge_point.name}_Cyclic_cost')[t]
                charge_point_cyclic_cost = sum(getattr(model, f'{charge_point.name}_ev{ev_index}_CycCost')[t] for ev_index in range(charge_point.num_evs))
                return cyclic_cost == charge_point_cyclic_cost
            setattr(self.model, f'{charge_point.name}_cyclic_cost_constraint', pyo.Constraint(self.model.T, rule=cyclic_cost))

            def calendar_cost(model, t, charge_point = charge_point):
                calendar_cost = getattr(model, f'{charge_point.name}_Calendar_cost')[t]
                charge_point_calendar_cost = sum(getattr(model, f'{charge_point.name}_ev{ev_index}_CalCost')[t] for ev_index in range(charge_point.num_evs))
                return calendar_cost == charge_point_calendar_cost
            setattr(self.model, f'{charge_point.name}_calendar_cost_constraint', pyo.Constraint(self.model.T, rule=calendar_cost))
        
        #Building constraints:
        for building in self.buildings:
            setattr(self.model, f'{building.name}_P', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000)))
            setattr(self.model, f'{building.name}_bess_soc', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, 1)))
            setattr(self.model, f'{building.name}_bess_ch', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, building.bess_max_power)))
            setattr(self.model, f'{building.name}_bess_ds', pyo.Var(self.model.T, within=pyo.Reals, bounds=(0, building.bess_max_power)))
            setattr(self.model, f'{building.name}_bess_Bch', pyo.Var(self.model.T, within=pyo.Binary))
            setattr(self.model, f'{building.name}_bess_CycAg', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_bess_CalAg', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_bess_CycCost', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_bess_CalCost', pyo.Var(self.model.T, within=pyo.Reals, bounds=(-1000000, 1000000), initialize = 0))
            setattr(self.model, f'{building.name}_bess_Ba', pyo.Var(self.model.T, within=pyo.Binary, initialize = 0))
            setattr(self.model, f'{building.name}_bess_Ka', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0, bounds=(0,1)))
            setattr(self.model, f'{building.name}_bess_Bb', pyo.Var(self.model.T, within=pyo.Binary, initialize = 0))
            setattr(self.model, f'{building.name}_bess_Kb', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0, bounds=(0,1)))
            setattr(self.model, f'{building.name}_bess_Bc', pyo.Var(self.model.T, within=pyo.Binary, initialize = 0))
            setattr(self.model, f'{building.name}_bess_Kc', pyo.Var(self.model.T, within=pyo.NonNegativeReals, initialize = 0, bounds=(0,1)))

            def bess_soc_rule(model, t, building = building):
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                if building.bess_capacity == 0:
                    return bess_soc == 0
                elif t == 0:
                    return bess_soc == building.bess_initial_soc + (bess_ch*building.efficiency - bess_ds/building.efficiency)/building.bess_capacity
                else:
                    bess_previous_soc = getattr(model, f'{building.name}_bess_soc')[t-1]
                    return bess_soc == bess_previous_soc + (bess_ch*building.efficiency - bess_ds/building.efficiency)/building.bess_capacity
            setattr(self.model, f'{building.name}_bess_soc_constraint', pyo.Constraint(self.model.T, rule = bess_soc_rule))

            def bess_soc_min_rule(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                if building.bess_capacity == 0:
                    return bess_soc == 0
                return bess_soc >= 0.2
            setattr(self.model, f'{building.name}_bess_soc_min_constraint', pyo.Constraint(self.model.T, rule = bess_soc_min_rule))  

            def bess_max_ch(model, t, building = building):
                bess_Bch = getattr(model, f'{building.name}_bess_Bch')[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                if building.bess_capacity == 0:
                    return bess_ch == 0
                return bess_ch <= bess_Bch*building.bess_max_power
            setattr(self.model, f'{building.name}_bess_max_ch_constraint', pyo.Constraint(self.model.T, rule = bess_max_ch))

            def bess_max_ds(model, t, building = building):
                bess_Bch = getattr(model, f'{building.name}_bess_Bch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                if building.bess_capacity == 0:
                    return bess_ds == 0
                return bess_ds <= (1-bess_Bch)*building.bess_max_power
            setattr(self.model, f'{building.name}_bess_max_ds_constraint', pyo.Constraint(self.model.T, rule = bess_max_ds))

            def building_consumption(model, t, building = building):
                P = getattr(model, f'{building.name}_P')[t]
                load = building.load[t]
                pv = building.pv_production[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                return load - pv - bess_ds + bess_ch == P
            setattr(self.model, f'{building.name}_consumption_constraint', pyo.Constraint(self.model.T, rule = building_consumption))

            #Aging Constraints

            def building_bess_cyclic_aging(model, t, building = building):
                cyclic_aging = getattr(model, f'{building.name}_bess_CycAg')[t]
                bess_ch = getattr(model, f'{building.name}_bess_ch')[t]
                bess_ds = getattr(model, f'{building.name}_bess_ds')[t]
                return cyclic_aging == (0.01*(((0.0000086*(self.temperature[t]**2)-0.0051*self.temperature[t]+0.763)*\
                                                      (67.15*bess_ch+67.15*bess_ds-2.94))/(building.bess_BatVol*building.bess_capacity)))
            setattr(self.model, f'{building.name}_bess_cyclic_aging_constraint', pyo.Constraint(self.model.T, rule = building_bess_cyclic_aging))

            def building_bess_cyclic_aging_cost(model, t, building = building):
                cyclic_aging = getattr(model, f'{building.name}_bess_CycAg')[t]
                cyclic_aging_cost = getattr(model, f'{building.name}_bess_CycCost')[t]
                return cyclic_aging_cost == (((1-building.bess_SalRep)*building.bess_Rep/((1+building.bess_discount)**building.bess_life)+building.bess_OM\
                                                *(((1+building.bess_discount)**building.bess_life)-1)/(building.bess_discount*(1+building.bess_discount)**building.bess_life))\
                                                /(building.bess_eoi))*(cyclic_aging)
            setattr(self.model, f'{building.name}_bess_cyclic_aging_cost_constraint', pyo.Constraint(self.model.T, rule=building_bess_cyclic_aging_cost))

            def building_bess_caldendar_aging(model, t, building = building):
                calendar_aging = getattr(model, f'{building.name}_bess_CalAg')[t]
                Ka = getattr(model, f'{building.name}_bess_Ka')[t]
                Ba = getattr(model, f'{building.name}_bess_Ba')[t]
                Kb = getattr(model, f'{building.name}_bess_Kb')[t]
                Bb = getattr(model, f'{building.name}_bess_Bb')[t]
                Kc = getattr(model, f'{building.name}_bess_Kc')[t]
                Bc = getattr(model, f'{building.name}_bess_Bc')[t]
                day = 1 #dummy day
                return calendar_aging == 0.01*((36.7*Ka + 1224.6*Ba) + (168.7*Kb + 3103.7*Bb) + (41.9*Kc + 6265.2*Bc))*(math.exp(-24500/(model.temperature[t]*8.314))*0.5*0.042)/((day+90)**0.5)
            setattr(self.model, f'{building.name}_building_bess_calendar_againg_constraint', pyo.Constraint(self.model.T, rule = building_bess_caldendar_aging))

            def buidling_bess_calendar_aging_cost(model, t, building = building):
                    calendar_aging = getattr(model, f'{building.name}_bess_CalAg')[t]
                    calendar_aging_cost = getattr(model, f'{building.name}_bess_CalCost')[t]
                    return calendar_aging_cost == (((1-building.bess_SalRep)*building.bess_Rep/((1+building.bess_discount)**building.bess_life)+building.bess_OM\
                                                *(((1+building.bess_discount)**building.bess_life)-1)/(building.bess_discount*(1+building.bess_discount)**building.bess_life))\
                                                /(building.bess_eoi))*(calendar_aging)
            setattr(self.model, f'{building.name}_building_besscalendar_aging_cost_constraint', pyo.Constraint(self.model.T, rule=buidling_bess_calendar_aging_cost))

            def building_bess_cal_aging_2(model, t, building = building):
                Ba = getattr(model, f'{building.name}_bess_Ba')[t]
                Bb = getattr(model, f'{building.name}_bess_Bb')[t]
                Bc = getattr(model, f'{building.name}_bess_Bc')[t]
                return Ba + Bb + Bc == 1
            setattr(self.model, f'{building.name}_bess_cal_aging_2_contraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_2))

            def building_bess_cal_aging_3(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                Ba = getattr(model, f'{building.name}_bess_Ba')[t]
                Bb = getattr(model, f'{building.name}_bess_Bb')[t]
                Bc = getattr(model, f'{building.name}_bess_Bc')[t]
                return bess_soc >= ((0.5*Bb) - (self.M*Ba) + (0.7*Bc))
            setattr(self.model, f'{building.name}_bess_cal_aging_3_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_3))

            def building_bess_cal_aging_4(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                Ba = getattr(model, f'{building.name}_bess_Ba')[t]
                Bb = getattr(model, f'{building.name}_bess_Bb')[t]
                Bc = getattr(model, f'{building.name}_bess_Bc')[t]
                return bess_soc <= ((0.5*Ba) + (self.M*Bc) + (0.7*Bb))
            setattr(self.model, f'{building.name}_bess_cal_aging_4_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_4))

            def building_bess_cal_aging_5(model, t, building = building):
                Ba = getattr(model, f'{building.name}_bess_Ba')[t]
                Ka = getattr(model, f'{building.name}_bess_Ka')[t]
                return Ka <= self.M*Ba
            setattr(self.model, f'{building.name}_bess_cal_aging_5_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_5))

            def building_bess_cal_aging_6(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                Ka = getattr(model, f'{building.name}_bess_Ka')[t]
                return Ka <= bess_soc
            setattr(self.model, f'{building.name}_bess_cal_aging_6_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_6))

            def building_bess_cal_aging_7(model, t, building = building):
                Ba = getattr(model, f'{building.name}_bess_Ba')[t]
                Ka = getattr(model, f'{building.name}_bess_Ka')[t]
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                return (bess_soc - (1-Ba) * self.M) <= Ka
            setattr(self.model, f'{building.name}_bess_cal_aging_7_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_7))

            def building_bess_cal_aging_8(model, t, building = building):
                Bb = getattr(model, f'{building.name}_bess_Bb')[t]
                Kb = getattr(model, f'{building.name}_bess_Kb')[t]
                return Kb <= self.M*Bb
            setattr(self.model, f'{building.name}_bess_cal_aging_8_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_8))

            def building_bess_cal_aging_9(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                Kb = getattr(model, f'{building.name}_bess_Kb')[t]
                return Kb <= bess_soc
            setattr(self.model, f'{building.name}_bess_cal_aging_9_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_9))

            def building_bess_cal_aging_10(model, t, building = building):
                Bb = getattr(model, f'{building.name}_bess_Bb')[t]
                Kb = getattr(model, f'{building.name}_bess_Kb')[t]
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                return (bess_soc - (1-Bb) * self.M) <= Kb
            setattr(self.model, f'{building.name}_bess_cal_aging_10_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_10))

            def building_bess_cal_aging_11(model, t, building = building):
                Bc = getattr(model, f'{building.name}_bess_Bc')[t]
                Kc = getattr(model, f'{building.name}_bess_Kc')[t]
                return Kc <= self.M*Bc
            setattr(self.model, f'{building.name}_bess_cal_aging_11_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_11))

            def building_bess_cal_aging_12(model, t, building = building):
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                Kc = getattr(model, f'{building.name}_bess_Kc')[t]
                return Kc <= bess_soc
            setattr(self.model, f'{building.name}_bess_cal_aging_12_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_12))

            def building_bess_cal_aging_13(model, t, building = building):
                Bc = getattr(model, f'{building.name}_bess_Bc')[t]
                Kc = getattr(model, f'{building.name}_bess_Kc')[t]
                bess_soc = getattr(model, f'{building.name}_bess_soc')[t]
                return (bess_soc - (1-Bc) * self.M) <= Kc
            setattr(self.model, f'{building.name}_bess_cal_aging_13_constraint', pyo.Constraint(self.model.T, rule=building_bess_cal_aging_13))

        def power_balance(model, t):
            overall_consumption = sum(getattr(model, f'{charge_point.name}_P')[t] for charge_point in self.charging_points) + \
                                    sum(getattr(model, f'{building.name}_P')[t] for building in self.buildings)
            P_im = self.model.P_im_grid[t]
            P_ex = self.model.P_ex_grid[t]
            return P_im - P_ex == overall_consumption
        self.model.power_balance_constarint = pyo.Constraint(self.model.T, rule=power_balance)

        def power_import(model, t):
            P_im = self.model.P_im_grid[t]
            B_im = self.model.B_im_grid[t]
            return P_im <= self.M * B_im
        self.model.power_import_constraint = pyo.Constraint(self.model.T, rule=power_import)

        def power_export(model, t):
            P_ex = self.model.P_ex_grid[t]
            B_im = self.model.B_im_grid[t]
            return P_ex <= self.M * (1 - B_im)
        self.model.power_export_constraint = pyo.Constraint(self.model.T, rule=power_export)

        def peak_load_constraint(model, t):
            return model.Peakload >= model.P_im_grid[t] - model.P_ex_grid[t]
        self.model.peak_load_constraint = pyo.Constraint(self.model.T, rule=peak_load_constraint)

        def previous_peak_check1(model):
            return model.monthly_peak >= model.Peakload
        self.model.previous_peak_check1_constraint = pyo.Constraint(rule=previous_peak_check1)

        def previous_peak_check2(model):
            return model.monthly_peak >= model.previous_monthly_peak
        self.model.previous_peak_check2_constraint = pyo.Constraint(rule=previous_peak_check2)

        def tranmission_cost(model, t):
            return model.transmission_cost[t] == model.P_im_grid[t] * model.Transmission_fee - model.P_ex_grid[t] * model.Transmission_health_incentive
        self.model.tranmission_cost_constraint = pyo.Constraint(self.model.T, rule = tranmission_cost)

        def supplier_cost_cal(model, t):
            return model.supplier_cost[t] == model.P_im_grid[t] * (model.spot_prices[t] + model.Energy_certificate) - model.P_ex_grid[t] * (model.spot_prices[t] + model.Energy_certificate + model.compensation_fee) 
        self.model.supplier_cost_constraint = pyo.Constraint(self.model.T, rule = supplier_cost_cal)

        def objective_rule(model):
            subscription_fee = model.Subscription_fee
            supplier_cost = sum((model.spot_prices[t] + model.Energy_certificate) * model.P_im_grid[t]/4 - (model.spot_prices[t] + model.Energy_certificate + model.compensation_fee) 
                                * model.P_ex_grid[t]/4 for t in model.T)
            transmission_cost = sum((model.Transmission_fee) * model.P_im_grid[t]/4 - (model.Transmission_health_incentive) 
                                * model.P_ex_grid[t]/4 for t in model.T)
            peak_cost = model.Effect_fee * model.monthly_peak * 5. #Check with "day/month" instead of 5
            dso_cost = transmission_cost + peak_cost + subscription_fee
            tax_cost = (supplier_cost + dso_cost)*0.25 + (1.25 * model.Energy_tax * sum(model.P_im_grid[t] - model.P_ex_grid[t] for t in model.T) / 4)
            ev_aging_cost = sum(sum(getattr(model, f'{charge_point.name}_Cyclic_cost')[t] + getattr(model, f'{charge_point.name}_Calendar_cost')[t] \
                                    for charge_point in self.charging_points) for t in model.T)
            building_bess_aging_cost = sum(sum(getattr(model, f'{building.name}_bess_CycCost')[t] #+ getattr(model, f'{building.name}_bess_CalCost')[t] \
                                    for building in self.buildings) for t in model.T)
            overall_cost = dso_cost + tax_cost + supplier_cost + ev_aging_cost + building_bess_aging_cost
            return overall_cost
        self.model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

    def solve(self):
        solver = pyo.SolverFactory('gurobi')
        self.results = solver.solve(self.model)
        return self.results

    def get_results(self):
        print(f'Objective value: {pyo.value(self.model.obj)}')
        print(f"⏱ Time steps in model: {len(self.model.T)}")
        results = {}
        for charge_point in self.charging_points:
            for ev_index in range(charge_point.num_evs):
                ev_name = f'{charge_point.name}_ev{ev_index}'
                results[f'{ev_name}_ch'] = [pyo.value(getattr(self.model, f'{ev_name}_ch')[t]) for t in self.model.T]
                results[f'{ev_name}_ds'] = [pyo.value(getattr(self.model, f'{ev_name}_ds')[t]) for t in self.model.T]
                results[f'{ev_name}_soc'] = [pyo.value(getattr(self.model, f'{ev_name}_soc')[t]) for t in self.model.T]
                results[f'{ev_name}_CalAg'] = [pyo.value(getattr(self.model, f'{ev_name}_CalAg')[t]) * 100 for t in self.model.T ]
                results[f'{ev_name}_CycAg'] = [pyo.value(getattr(self.model, f'{ev_name}_CycAg')[t]) * 100 for t in self.model.T]
            results[f'{charge_point.name}_P'] = [pyo.value(getattr(self.model, f'{charge_point.name}_P')[t]) for t in self.model.T]
        for building in self.buildings:
            results[f'{building.name}_bess_ch'] = [pyo.value(getattr(self.model, f'{building.name}_bess_ch')[t]) for t in self.model.T]
            results[f'{building.name}_bess_ds'] = [pyo.value(getattr(self.model, f'{building.name}_bess_ds')[t]) for t in self.model.T]
            results[f'{building.name}_bess_soc'] = [pyo.value(getattr(self.model, f'{building.name}_bess_soc')[t]) for t in self.model.T]
            results[f'{building.name}_P'] = [pyo.value(getattr(self.model, f'{building.name}_P')[t]) for t in self.model.T]
            results[f'{building.name}_bess_CycAg'] = [pyo.value(getattr(self.model, f'{building.name}_bess_CycAg')[t]) * 100 for t in self.model.T]
            results[f'{building.name}_bess_CycCost'] = [pyo.value(getattr(self.model, f'{building.name}_bess_CycCost')[t]) for t in self.model.T]
            results[f'{building.name}_bess_CalAg'] = [pyo.value(getattr(self.model, f'{building.name}_bess_CalAg')[t]) * 100 for t in self.model.T]
            results[f'{building.name}_bess_CalCost'] = [pyo.value(getattr(self.model, f'{building.name}_bess_CalCost')[t]) for t in self.model.T]
        results['P_import'] = [pyo.value(self.model.P_im_grid[t]) for t in self.model.T]
        results['P_export'] = [pyo.value(self.model.P_ex_grid[t]) for t in self.model.T]
        results['Transmission cost'] = [pyo.value(self.model.transmission_cost[t]) for t in self.model.T]
        results['Supplier cost'] = [pyo.value(self.model.supplier_cost[t]) for t in self.model.T]
        return pd.DataFrame(results)

In [3]:
#name, ev_capacity, ev_max_power, ev_arrival_soc, ev_arrival, ev_departure, ev_desired_soc
T = 24

# Spot prices (€/kWh)
spot_prices = [0.0 if i < 6 or i > 18 else 0.5 + 0.01*np.sin(i*np.pi/12) for i in range(T)]

# 24-hour realistic load and PV
load = [15 + 0.5*np.sin(i*np.pi/12) for i in range(T)]
pv_production = [0.0 if i < 6 or i > 18 else 1.5*np.sin((i-6)*np.pi/12) for i in range(T)]
load1 = [10 + 0.5*np.sin(i*np.pi/12) for i in range(T)]
pv_production1 = [0.0 if i < 6 or i > 18 else 2.5*np.sin((i-6)*np.pi/12) for i in range(T)]

base_temp = -5  # average daily temperature
amplitude = 4   # daily variation

winter_temperature = [
    base_temp + amplitude * np.cos((i - 8) * np.pi / 12)  # coldest at ~5 AM
    + np.random.normal(0, 0.5) + 273  # small natural disturbance
    for i in range(T)
]

cp1 = charging_point(name = 'cp1', ev_capacity=[45, 65], ev_max_power=[10, 12], ev_arrival=[6, 18], ev_departure= [12, 23], ev_arrival_soc=[0.5, 0.3], ev_desired_soc=[0.75, 0.6])
cp2 = charging_point(name = 'cp2', ev_capacity=[55, 95], ev_max_power=[10, 12], ev_arrival=[8, 15], ev_departure= [12, 20], ev_arrival_soc=[0.4, 0.7], ev_desired_soc=[0.75, 0.75])

b1 = building(name= 'b1', load = load, pv_production=pv_production, bess_capacity=100, bess_initial_soc=0.5, bess_max_power=15)
b2 = building(name= 'b2', load = load1, pv_production=pv_production1, bess_capacity=80, bess_initial_soc=0.5, bess_max_power=7.5)
b3 = building(name= 'b3', load = [0*i for i in range(len(load1))], pv_production=[0*i for i in range(len(pv_production1))], bess_capacity=80, bess_initial_soc=0.5, bess_max_power=7.5)

opt_model = LEC_Opt_spot([cp1, cp2], [b1, b2, b3], spot_prices, temperature=winter_temperature)
results_df = opt_model.solve()
df = opt_model.get_results()

Objective value: 493.0557718426812
⏱ Time steps in model: 24


In [4]:
df

,cp1_ev0_ch,cp1_ev0_ds,cp1_ev0_soc,cp1_ev0_CalAg,cp1_ev0_CycAg,cp1_ev1_ch,cp1_ev1_ds,cp1_ev1_soc,cp1_ev1_CalAg,cp1_ev1_CycAg,...,b3_bess_soc,b3_P,b3_bess_CycAg,b3_bess_CycCost,b3_bess_CalAg,b3_bess_CalCost,P_import,P_export,Transmission cost,Supplier cost
0,0.000000,0.0,0.00,0.000042,0.000000,0.000000,0.0,0.000000,0.000042,0.000000,...,0.500000,0.000000,-0.000001,-0.003826,0.000109,0.305736,23.219707,0.0,2.623827,0.116099
1,0.000000,0.0,0.00,0.000042,0.000000,0.000000,0.0,0.000000,0.000042,0.000000,...,0.500000,0.000000,-0.000001,-0.003782,0.000110,0.309967,23.219707,0.0,2.623827,0.116099
2,0.000000,0.0,0.00,0.000047,0.000000,0.000000,0.0,0.000000,0.000047,0.000000,...,0.500000,0.000000,-0.000001,-0.003474,0.000048,0.133406,23.219707,0.0,2.623827,0.116099
3,0.000000,0.0,0.00,0.000048,0.000000,0.000000,0.0,0.000000,0.000048,0.000000,...,0.500000,0.000000,-0.000001,-0.003401,0.000049,0.136708,23.219707,0.0,2.623827,0.116099
4,0.000000,0.0,0.00,0.000050,0.000000,0.000000,0.0,0.000000,0.000050,0.000000,...,0.500000,0.000000,-0.000001,-0.003308,0.000050,0.141127,23.219707,0.0,2.623827,0.116099
5,0.000000,0.0,0.00,0.000051,0.000000,0.000000,0.0,0.000000,0.000051,0.000000,...,0.500000,0.000000,-0.000001,-0.003207,0.000052,0.146251,23.219707,0.0,2.623827,0.116099
6,0.000000,0.0,0.50,0.000053,0.000000,0.000000,0.0,0.000000,0.000052,0.000000,...,0.500000,0.000000,-0.000001,-0.003170,0.000053,0.148188,23.219707,0.0,2.623827,11.958149
7,0.000000,0.0,0.50,0.000052,-0.000002,0.000000,0.0,0.000000,0.000052,0.000000,...,0.500000,0.000000,-0.000001,-0.003196,0.000052,0.146819,23.219707,0.0,2.623827,11.950237
8,0.000000,0.0,0.50,0.000055,-0.000002,0.000000,0.0,0.000000,0.000055,0.000000,...,0.491313,-0.646319,0.000015,0.041835,0.000055,0.155389,23.219707,0.0,2.623827,11.927040
9,0.000000,0.0,0.50,0.000053,-0.000002,0.000000,0.0,0.000000,0.000052,0.000000,...,0.416408,-5.572951,0.000142,0.399508,0.000053,0.148159,23.219707,0.0,2.623827,11.890140


In [8]:
(df['b1_bess_CalAg'].sum() + df['b1_bess_CycAg'].sum())*365

np.float64(0.6420454598118642)